In [53]:
import os
os.environ["KERAS_BACKEND"] = "torch"
import torch
from torch import nn
import keras
from keras.layers import TorchModuleWrapper
from torch.nn import functional as F
import numpy as np

In [54]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [55]:
# Define the dimensions based on the model
batch_size = 64
input_features = 32
num_classes = 10
# 1. Generate random input data (x)
# Shape: (batch_size, input_features) -> e.g., (64, 32)
# Using float32 is standard for deep learning frameworks.
x_test = np.random.rand(batch_size, input_features).astype('float32')

# 2. Generate random integer labels (y)
# Shape: (batch_size,) -> e.g., (64,)
# The labels are integers from 0 to 9 for 10 classes.
y_test = np.random.randint(0, num_classes, size=batch_size)

print("Shape of x_test:", x_test.shape)
print("Shape of y_test:", y_test.shape)
print("Sample labels:", y_test[:5])

Shape of x_test: (64, 32)
Shape of y_test: (64,)
Sample labels: [5 2 4 6 1]


In [68]:
class PyTorchModelWithBatchNorm(nn.Module):
    def __init__(self, input_features, output_features):
        super().__init__()
        self.dense1 = nn.Linear(input_features, 64)
        self.bn1 = nn.BatchNorm1d(64) # BatchNorm layer
        self.relu = nn.ReLU()
        self.dense2 = nn.Linear(64, output_features)

    # CRITICAL: The forward method must accept a 'training' argument.
    def forward(self, x, taining=False):
        # Use the flag to set the mode for BatchNorm/Dropout
        if taining:
            self.train()
        else:
            self.eval()

        x = self.dense1(x)
        x = self.bn1(x) # BatchNorm will now behave correctly
        x = self.relu(x)
        x = self.dense2(x)
        return x

In [69]:
# Instantiate the PyTorch model
pytorch_model = PyTorchModelWithBatchNorm(input_features=32, output_features=10)
pytorch_model.eval()

PyTorchModelWithBatchNorm(
  (dense1): Linear(in_features=32, out_features=64, bias=True)
  (bn1): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU()
  (dense2): Linear(in_features=64, out_features=10, bias=True)
)

In [71]:
pytorch_model(torch.from_numpy(x_test[:1]))

tensor([[ 0.0944,  0.3796,  0.2515, -0.1226,  0.1134, -0.1798,  0.1615,  0.2097,
         -0.0890,  0.2003]], grad_fn=<AddmmBackward0>)

In [72]:
# Wrap it in a single step
wrapped_model_block = TorchModuleWrapper(pytorch_model)

# Build the final Keras model
model = keras.Sequential([
    keras.layers.Input(shape=(32,)),
    wrapped_model_block
])

model.compile(optimizer="adam", loss="sparse_categorical_crossentropy")
model.summary()

Model: "sequential_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ torch_module_wrapper_13         │ (None, 10)             │         2,890 │
│ (TorchModuleWrapper)            │                        │               │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,890 (11.29 KB)

 Trainable params: 2,890 (11.29 KB)

 Non-trainable params: 0 (0.00 B)

In [73]:
pytorch_model(torch.from_numpy(x_test[:1]).to(device))

tensor([[ 0.0944,  0.3796,  0.2515, -0.1226,  0.1134, -0.1798,  0.1615,  0.2097,
         -0.0890,  0.2003]], device='cuda:0', grad_fn=<AddmmBackward0>)

In [74]:
model(x_test[:1])

tensor([[ 0.0944,  0.3796,  0.2515, -0.1226,  0.1134, -0.1798,  0.1615,  0.2097,
         -0.0890,  0.2003]], device='cuda:0', grad_fn=<AddmmBackward0>)

In [ ]:
model.train()

In [62]:
input_tensor = torch.from_numpy(x_test[:1]).to(device)
input_tensor.requires_grad_(True)
# 2. Effectuez la passe avant (forward pass)
output = pytorch_model(input_tensor)

# 3. Calculez les gradients (backward pass)
# .backward() ne fonctionne que sur un scalaire.
# On somme donc la sortie pour en faire un scalaire.
# Cela calcule le gradient de la somme des sorties par rapport à l'entrée.
output.sum().backward()

In [63]:
gradient = input_tensor.grad
print("Gradient :")
print(gradient)


Gradient :
tensor([[ 0.1737, -0.3116,  0.0494, -0.0126,  0.0577,  0.2042,  0.1719,  0.1899,
         -0.0654, -0.1522, -0.2112,  0.0829, -0.0965, -0.0868, -0.2439, -0.0918,
          0.2775, -0.0436, -0.1420,  0.0361, -0.0354,  0.0417,  0.1756,  0.2714,
          0.0837, -0.0462, -0.1646, -0.0386, -0.0018,  0.0055,  0.1286,  0.0675]],
       device='cuda:0')


In [64]:
input_tensor = torch.from_numpy(x_test[:1]).to(device)
input_tensor.requires_grad_(True)
# 2. Effectuez la passe avant (forward pass)
output = model(input_tensor)

# 3. Calculez les gradients (backward pass)
# .backward() ne fonctionne que sur un scalaire.
# On somme donc la sortie pour en faire un scalaire.
# Cela calcule le gradient de la somme des sorties par rapport à l'entrée.
output.sum().backward()

In [65]:
gradient = input_tensor.grad
print("Gradient :")
print(gradient)


Gradient :
tensor([[ 0.1737, -0.3116,  0.0494, -0.0126,  0.0577,  0.2042,  0.1719,  0.1899,
         -0.0654, -0.1522, -0.2112,  0.0829, -0.0965, -0.0868, -0.2439, -0.0918,
          0.2775, -0.0436, -0.1420,  0.0361, -0.0354,  0.0417,  0.1756,  0.2714,
          0.0837, -0.0462, -0.1646, -0.0386, -0.0018,  0.0055,  0.1286,  0.0675]],
       device='cuda:0')


In [66]:
model.summary()

Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ torch_module_wrapper_12         │ (None, 10)             │         2,890 │
│ (TorchModuleWrapper)            │                        │               │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,890 (11.29 KB)

 Trainable params: 2,890 (11.29 KB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
# This is our custom adapter module
class PretrainedModelWrapper(nn.Module):
    def __init__(self, pretrained_model):
        super().__init__()
        # Load the pre-trained model inside the wrapper
        self.pretrained_model = pretrained_model
    # Implement the Keras-compatible forward method
    def forward(self, x, training=False):
        # 1. Use the training flag to set the mode
        if training:
            self.pretrained_model.train()
        else:
            self.pretrained_model.eval()

        # 2. Call the original model's forward pass
        return self.pretrained_model(x)